# Antasena Super Course — Perbandingan Efisiensi (Latihan)

**Tujuan:** membandingkan dua data telemetri dan mencari tahu *mengapa* salah satunya lebih hemat energi.

Setiap kali kamu menemukan `___` di sel kode, ganti dengan kode yang benar lalu jalankan selnya. Sel yang bertanda **✏️ TUGAS** perlu kamu isi. Petunjuk ada di teks di atas setiap sel.

Jalankan sel dari atas ke bawah, karena sel di bawah bergantung pada sel sebelumnya.

## Persiapan (khusus Google Colab)

Jika kamu membuka notebook ini di **Google Colab**, jalankan sel di bawah terlebih dahulu. Sel ini mengunduh data dan skrip kursus ke Colab. Di Jupyter lokal, sel ini tidak melakukan apa-apa.

In [ ]:
import os, sys

if "google.colab" in sys.modules and not os.path.exists("data"):
    if not os.path.exists("/content/ASC_Simulator"):
        !git clone -q https://github.com/Antasena-ITS-Team/ASC_Simulator.git /content/ASC_Simulator
    %cd /content/ASC_Simulator

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Konfigurasi

Setiap baris berisi `(path_csv, label)`.

In [ ]:
ATTEMPTS = [
    ("data/DataASC_105V_9A.csv", "105V / 9A"),
    ("data/DataASC_80V_12A.csv", "80V / 12A"),
]

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # aman untuk buta warna dengan urutan ini
COLORS = {label: PALETTE[i % len(PALETTE)] for i, (_, label) in enumerate(ATTEMPTS)}

## Langkah 1 — Memuat data dan menghitung besaran turunan

- `speed_ms` — jika file menyimpan `speed_kmh`, ubah satuannya (**1 m/s = 3,6 km/jam**)
- `power_W = voltage_V × current_A` (daya = tegangan × arus)
- `dt_s` — selang waktu antar baris (`.diff()`)
- `energy_Wh` — jumlah kumulatif `power_W × dt_s`, diubah dari joule (W·s) ke Wh (**1 Wh = 3600 J**)
- `efficiency_km_per_kWh = distance_m / energy_Wh` — kebetulan m/Wh dan km/kWh bernilai sama

**Cek kewajaran:** *nama* kolom belum tentu menjamin *satuannya*. `check_speed_units` membandingkan kolom kecepatan dengan kecepatan yang dihitung dari `distance_m` dan `time_s`. Jika keduanya berbeda 3,6 kali, berarti kolom itu sebenarnya dalam km/jam.

✏️ **TUGAS:** isi keempat bagian kosong di `load_attempt`.

In [ ]:
def check_speed_units(df, label):
    implied = df["distance_m"].diff() / df["time_s"].diff()
    moving = df["speed_ms"] > 1
    ratio = (df.loc[moving, "speed_ms"] / implied[moving]).median()
    if abs(ratio - 3.6) < 0.1:
        print(f"[{label}] speed_ms bernilai {ratio:.2f}x kecepatan dari jarak/waktu -> sebenarnya km/jam, dikonversi")
        df["speed_ms"] = df["speed_ms"] / 3.6
    return df


def load_attempt(path, label):
    df = pd.read_csv(path)

    if "speed_kmh" in df.columns:
        df["speed_ms"] = df["speed_kmh"] / ___
    df = check_speed_units(df, label)

    df["power_W"] = ___
    df["dt_s"] = df["time_s"].diff().fillna(0)
    df["energy_Wh"] = (df["power_W"] * df["dt_s"] / ___).cumsum()
    df["efficiency_km_per_kWh"] = np.where(
        df["energy_Wh"] > 0, ___, np.nan
    )
    df["attempt"] = label
    return df


data = {label: load_attempt(path, label) for path, label in ATTEMPTS}

for label, df in data.items():
    print(f"--- {label} ---")
    display(df.head(3))

## Langkah 2 — Sinyal mentah terhadap waktu

`plot_over_time(column, ylabel, title)` menggambar satu garis untuk setiap percobaan.

✏️ **TUGAS:** isi data sumbu x dan y untuk `plt.plot`, lalu lengkapi pemanggilan untuk arus dan kecepatan.

In [ ]:
def plot_over_time(column, ylabel, title):
    plt.figure(figsize=(9, 4.5))
    for label, df in data.items():
        plt.plot(___, ___, label=label, color=COLORS[label], linewidth=1.2)
    plt.xlabel("waktu (s)")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_over_time("voltage_V", "tegangan (V)", "Tegangan baterai terhadap waktu")

In [ ]:
plot_over_time(___, "arus (A)", "Arus terhadap waktu")

In [ ]:
plot_over_time(___, ___, "Kecepatan terhadap waktu")

## Langkah 3 — Daya dan energi

Pada grafik energi terhadap jarak, **garis yang lebih landai = jarak per Wh lebih jauh = lebih efisien.**

✏️ **TUGAS:** plot daya terhadap waktu, lalu isi kolom x/y untuk grafik energi.

In [ ]:
plot_over_time(___, "daya (W)", "Daya terhadap waktu")

In [ ]:
plt.figure(figsize=(9, 4.5))
for label, df in data.items():
    plt.plot(df[___], df[___], label=label, color=COLORS[label], linewidth=1.5)
plt.xlabel("jarak (m)")
plt.ylabel("energi kumulatif (Wh)")
plt.title("Energi yang dipakai per jarak tempuh (lebih landai = lebih efisien)")
plt.legend()
plt.tight_layout()
plt.show()

## Langkah 4 — Perbandingan efisiensi

Efisiensi kumulatif masih naik-turun di awal (karena dibagi energi yang masih sangat kecil) lalu stabil seiring berjalannya balapan. Bandingkan nilai **akhirnya**.

✏️ **TUGAS:** ambil nilai terakhir kolom efisiensi. Petunjuk: `.iloc[-1]`.

In [ ]:
plt.figure(figsize=(9, 4.5))
for label, df in data.items():
    plt.plot(df["distance_m"], df["efficiency_km_per_kWh"], label=label, color=COLORS[label], linewidth=1.2)
plt.xlabel("jarak (m)")
plt.ylabel("efisiensi (km/kWh)")
plt.title("Efisiensi kumulatif sepanjang balapan")
plt.ylim(0, 150)
plt.legend()
plt.tight_layout()
plt.show()

for label, df in data.items():
    final_eff = ___
    print(f"{label}: efisiensi akhir = {final_eff:.2f} km/kWh")

### Ringkasan per lap

Untuk setiap lap: jarak tempuh = `distance_m` terakhir − pertama; begitu juga untuk `energy_Wh`.

✏️ **TUGAS:** hitung `lap_distance_m`, `lap_energy_Wh`, dan `lap_efficiency_km_per_kWh`.

In [ ]:
summaries = []
for label, df in data.items():
    g = df.groupby("lap").agg(
        distance_start=("distance_m", "first"), distance_end=("distance_m", "last"),
        energy_start=("energy_Wh", "first"), energy_end=("energy_Wh", "last"),
        avg_speed_ms=("speed_ms", "mean"), max_speed_ms=("speed_ms", "max"),
        avg_current_A=("current_A", "mean"),
    )
    g["lap_distance_m"] = ___
    g["lap_energy_Wh"] = ___
    g["lap_efficiency_km_per_kWh"] = ___
    g["attempt"] = label
    summaries.append(g.reset_index()[[
        "attempt", "lap", "lap_distance_m", "lap_energy_Wh",
        "lap_efficiency_km_per_kWh", "avg_speed_ms", "max_speed_ms", "avg_current_A",
    ]])

lap_summary = pd.concat(summaries, ignore_index=True)
lap_summary.round(2)

In [ ]:
pivot = lap_summary.pivot(index="lap", columns="attempt", values="lap_efficiency_km_per_kWh")
pivot.plot(kind="bar", color=[COLORS[c] for c in pivot.columns], figsize=(8, 4.5))
plt.ylabel("efisiensi (km/kWh)")
plt.title("Efisiensi per lap")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Langkah 5 — Mengapa efisiensinya berbeda?

Tabel di bawah mengumpulkan angka-angka yang kamu perlukan untuk menjawab pertanyaan.

In [ ]:
overall = pd.DataFrame({
    label: {
        "efisiensi akhir (km/kWh)": df["efficiency_km_per_kWh"].iloc[-1],
        "total energi (Wh)": df["energy_Wh"].iloc[-1],
        "rata-rata tegangan (V)": df["voltage_V"].mean(),
        "rata-rata arus (A)": df["current_A"].mean(),
        "rata-rata arus² (A²)": (df["current_A"] ** 2).mean(),
        "rata-rata kecepatan (m/s)": df["speed_ms"].mean(),
        "kecepatan maks (m/s)": df["speed_ms"].max(),
        "waktu tempuh (s)": df["time_s"].iloc[-1],
    }
    for label, df in data.items()
})
overall.round(2)

In [ ]:
plt.figure(figsize=(9, 4.5))
for label, df in data.items():
    plt.scatter(df["speed_ms"], df["current_A"], label=label, color=COLORS[label], s=6, alpha=0.4)
plt.xlabel("kecepatan (m/s)")
plt.ylabel("arus (A)")
plt.title("Arus terhadap kecepatan")
plt.legend()
plt.tight_layout()
plt.show()

### P1. Percobaan mana yang efisiensinya lebih tinggi, dan kira-kira berapa persen selisihnya?

✏️ _Tulis jawabanmu di sini._

### P2. Bandingkan `avg_current_A` antar percobaan. Rugi-rugi listrik sebanding dengan **kuadrat arus** (`P_rugi ≈ I² × R`). Bagaimana hal ini bisa menjelaskan sebagian perbedaannya?

✏️ _Tulis jawabanmu di sini._

### P3. Apakah percobaan yang lebih efisien juga mencapai kecepatan puncak lebih tinggi? Apa artinya bagi tarik-ulur antara kecepatan dan efisiensi?

✏️ _Tulis jawabanmu di sini._

### P4. Lihat grafik arus terhadap kecepatan: pada kecepatan yang mirip, apakah salah satu percobaan menarik arus jauh lebih besar? Apa penyebabnya (rasio gir, ukuran motor, hambatan aerodinamis, gaya mengemudi)?

✏️ _Tulis jawabanmu di sini._

## Langkah 6 — Giliranmu

1. Jalankan `publisher.py` untuk satu data rekaman sementara `subscriber.py --out attempt_a.csv` sedang mendengarkan; ulangi untuk percobaan kedua ke `attempt_b.csv`.
2. Ubah `ATTEMPTS` di sel konfigurasi agar menunjuk ke kedua file tersebut.
3. Kernel → Restart & Run All, lalu tuliskan penjelasanmu sendiri.

### Memakai data rekamanmu sendiri di Colab

Setiap notebook Colab berjalan di mesinnya sendiri, jadi file yang direkam di notebook lain **tidak otomatis ada di sini**. Pilih salah satu cara:

- **Cara A:** rekam dengan notebook [rekam_data.ipynb](https://colab.research.google.com/github/Antasena-ITS-Team/ASC_Simulator/blob/master/rekam_data.ipynb), unduh CSV-nya, lalu jalankan sel unggah di bawah.
- **Cara B:** rekam langsung di notebook ini dengan sel rekam di bawah (jalankan saat instruktur sedang menyiarkan data).

Setelah itu, ubah `ATTEMPTS` di sel konfigurasi ke nama file-mu dan jalankan ulang semua sel. Di komputer lokal, cukup jalankan `python subscriber.py` di terminal seperti di README.

⚠️ Di Colab, file akan **hilang saat runtime berakhir**. Unduh CSV-mu lewat panel **Files** di kiri.

In [ ]:
# Cara A: unggah CSV hasil rekaman (khusus Colab)
if "google.colab" in sys.modules:
    from google.colab import files
    uploaded = files.upload()
    print("Tersimpan:", list(uploaded))

In [ ]:
# Cara B: rekam langsung di sini (khusus Colab)
!pip install -q paho-mqtt

OUT_FILE = "attempt_a.csv"   # ganti ke attempt_b.csv untuk percobaan kedua
DURASI_S = 300              # berapa detik merekam
!timeout {DURASI_S} python -u subscriber.py --out {OUT_FILE} | grep -v "^Received"